In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv('../data/train.csv').drop(columns='ID')
X_train = train.drop(columns='y')
y_train = train['y']

X_test = pd.read_csv('../data/test.csv').drop(columns='ID')

In [3]:
train.head(20)

,x_0,x_1,x_2,x_3,x_4,x_5,x_6,x_7,x_8,x_9,x_10,y
0,1.006187,-1.962566,1.247535,0.926500,-0.265766,-1.789301,0.470004,-0.139467,0.623996,0.320359,0.078612,83.424500
1,1.024647,-2.472625,1.144386,0.846499,-0.287336,-1.756679,0.503860,-0.219545,0.697607,0.238306,0.081778,79.374109
2,1.062444,-2.451003,1.186546,0.873599,-0.257828,-1.802735,0.498045,-0.194247,0.684134,0.259392,0.095003,82.181616
3,1.089189,-2.458470,1.184531,0.810867,-0.276517,-1.787739,0.503359,-0.201923,0.686394,0.245736,0.091737,83.006586
4,1.023323,-2.133468,1.242266,0.939837,-0.264515,-1.792044,0.470478,-0.142896,0.623778,0.314610,0.078987,83.051434
5,1.022329,-1.908444,1.230314,0.933263,-0.285966,-1.752026,0.475435,-0.129337,0.599161,0.341141,0.060320,85.706116
6,0.926335,-1.308083,1.158176,0.946554,-0.401927,-1.642515,0.511611,-0.086523,0.580449,0.384201,0.008076,90.609204
7,1.031117,-2.516612,1.124304,0.818973,-0.295093,-1.754170,0.507188,-0.214431,0.691660,0.235866,0.077905,84.635248
8,1.052547,-2.471300,1.158063,0.785188,-0.284571,-1.767290,0.506143,-0.210996,0.692922,0.234848,0.084261,80.978638
9,1.053818,-2.392457,1.204323,0.880342,-0.268666,-1.801167,0.496331,-0.184487,0.683482,0.260503,0.095857,82.369635


In [37]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train, X_test = scaler.fit_transform(X_train), scaler.transform(X_test)
X_train, X_test  = pd.DataFrame(X_train), pd.DataFrame(X_test)

In [48]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
from torch.utils.data import DataLoader, TensorDataset

X_train_t, y_train_t = torch.tensor(X_train.values, dtype=torch.float32), torch.tensor(y_train.values, dtype=torch.float32)
train_loader = DataLoader(TensorDataset(X_train_t,y_train_t), batch_size=32, shuffle=True)

In [49]:
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim, hidden_dim=128, dropout_rate = 0.1):
        super(VAE, self).__init__()
        
        # Encoder
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2_mean = nn.Linear(hidden_dim, latent_dim)
        self.fc2_logvar = nn.Linear(hidden_dim, latent_dim)
        
        # Decoder
        self.fc3 = nn.Linear(latent_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, input_dim)
        
        # Dropout for uncertainty estimate
        self.dropout = nn.Dropout(p = dropout_rate)
    
    def encode(self, x):
        h1 = F.relu(self.fc1(x))
        return self.fc2_mean(h1), self.fc2_logvar(h1)
        
    def reparameterize(self, mu , logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps*std
    
    def decode(self, z):
        h3 = F.relu(self.fc3(z))
        h3 = self.dropout(h3)
        return torch.sigmoid(self.fc4(h3))
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar
    
    def loss_function(self, recon_x, x, mu, logvar):
        BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
        KLD = -0.5 * torch.sum(1+logvar-mu.pow(2)-logvar.exp())
        return BCE+KLD

In [50]:
# estimate decoder uncertainty
def estimate_uncertainty(vae, z, n_samples= 10):
    vae.train()
    predictions = []
    
    for _ in range(n_samples):
        predictions.append(vae.decode(z))
        
    predictions = torch.stack(predictions)
    uncertainty = torch.var(predictions, dim=0)
    
    return uncertainty

In [51]:
def optimize_in_latent_space(vae, target_function, latent_dim, n_iterations=100):
    z = torch.randn(1, latent_dim, requires_grad=True)
    optimizer = optim.Adam([z], lr=0.01)
    
    for i in range(n_iterations):
        vae.eval()
        uncertainty = estimate_uncertainty(vae, z)
        pred = vae.decode(z)
        loss = -target_function(pred) + torch.mean(uncertainty)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    return z

In [55]:
def train_vae(vae, data_loader, n_epochs=100):
    optimizer = torch.optim.Adam(vae.parameters(), lr= 0.001)
    
    for epoch in range(n_epochs):
        vae.train()
        train_loss = 0
        
        for data in data_loader:
            optimizer.zero_grad()
            recon_batch, mu, logvar = vae(data)
            loss = vae.loss_function(recon_batch, data, mu, logvar)
            loss.backward()
            train_loss += loss.item()
            optimizer.step()
            
        print(f'Epoch : {epoch}, Loss : P{train_loss/len(data_loader.dataset)}')

vae = VAE(input_dim=X_train_t.shape, latent_dim = 3)

train_vae(vae, train_loader)

TypeError: empty(): argument 'size' failed to unpack the object at pos 2 with error "type must be tuple of ints,but got torch.Size"